# Problem 69: Circuit Breaker Pattern

**Difficulty:** Medium | **Framework:** LangGraph

**Categories:** error-recovery, orchestration

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/circuit-breaker-pattern).

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The circuit breaker must open after 3 consecutive failures
- Once open, the breaker must reject calls for 60 seconds without hitting the service
- After the cooldown, the breaker must allow one trial call to check recovery
- The breaker state must be shared across all calls to the same service


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
import time
import random
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class State(TypedDict):
    amount: str
    result: str
    attempt: int

def process_payment(state: State) -> State:
    """Process a payment through the payment service."""
    # BUG: The service is down and there is no circuit breaker
    raise ConnectionError("Payment service is unavailable")

graph = StateGraph(State)
graph.add_node("payment", process_payment)
graph.add_edge(START, "payment")
graph.add_edge("payment", END)

app = graph.compile()

# Test: The agent keeps hitting a dead service with no protection
for i in range(5):
    print(f"\nAttempt {i+1}:")
    result = app.invoke({"amount": "$50", "result": "", "attempt": i})
    print(result["result"])

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Track failure count and last failure timestamp in a class or shared state object
2. Implement three states: CLOSED (normal), OPEN (rejecting), HALF-OPEN (testing recovery)
3. In HALF-OPEN state, let one call through — if it succeeds, reset to CLOSED; if it fails, go back to OPEN


## 7. Evaluation Criteria

Check your solution against these criteria:

- Opens after 3 consecutive failures
- Rejects calls when open
- Tests recovery in half-open state
- Resets on successful recovery
